In [1]:
import pandas as pd
df=pd.read_csv(r"C:\Users\bisha\Downloads\malicious_phish.csv\malicious_phish.csv")

In [2]:
df.head()

,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


In [3]:
df.columns

Index(['url', 'type'], dtype='object')

In [4]:
df['type'].value_counts()

type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

In [5]:
#Mapping of type with integer
lap_map={
    'benign':0,
    'defacement':1,
    'phishing':2,
    'malware':3
}
df['new_l']=df['type'].map(lap_map)

In [6]:
df.head()

,url,type,new_l
0,br-icloud.com.br,phishing,2
1,mp3raid.com/music/krizz_kaliko.html,benign,0
2,bopsecrets.org/rexroth/cr/1.htm,benign,0
3,http://www.garage-pirenne.be/index.php?option=...,defacement,1
4,http://adventure-nicaragua.net/index.php?optio...,defacement,1


In [7]:
df.isnull().sum()

url      0
type     0
new_l    0
dtype: int64

In [8]:
df_small=df.sample(n=50000,random_state=42)

In [9]:
#Train_test split
from sklearn.model_selection import train_test_split
train_df,val_df=train_test_split(
    df_small,
    test_size=0.2,
    stratify=df_small['new_l'],
    random_state=42
)

In [10]:
#Model creation and tokenization
from transformers import BertTokenizerFast
tokenizer=BertTokenizerFast.from_pretrained(r"C:\bert-tokenizer")

In [11]:
def tokenize_urls(urls,max_length=128):
    return tokenizer(
        urls,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        batch_size=1000,
        return_tensors="pt"
    )

In [12]:
train_encodings = tokenize_urls(train_df['url'].tolist())
val_encodings   = tokenize_urls(val_df['url'].tolist())

In [13]:
train_encodings.keys()

KeysView({'input_ids': tensor([[ 101, 2111, 1012,  ...,    0,    0,    0],
        [ 101, 2012, 2102,  ...,    0,    0,    0],
        [ 101, 8299, 1024,  ...,    0,    0,    0],
        ...,
        [ 101, 5906, 1012,  ...,    0,    0,    0],
        [ 101, 6697, 1012,  ...,    0,    0,    0],
        [ 101, 8299, 1024,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])})

In [14]:
import torch
train_labels=torch.tensor(train_df['new_l'].values)
val_labels=torch.tensor(val_df['new_l'].values)

In [15]:
from torch.utils.data import Dataset

class URLDataset(Dataset):
    def __init__(self,encodings,labels):
        self.encodings=encodings
        self.labels=labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self,idx):
        item={key:val[idx] for key,val in self.encodings.items()}
        item['labels']=self.labels[idx]
        return item

In [16]:
train_dataset=URLDataset(train_encodings,train_labels)
val_dataset=URLDataset(val_encodings,val_labels)

### Training of model

In [17]:
from transformers import BertForSequenceClassification
model=BertForSequenceClassification.from_pretrained(r"C:\bert-model",num_labels=train_df['new_l'].nunique())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: C:\bert-model
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your

In [18]:
import torch
device=torch.device("cuda"if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [19]:
from torch.optim import AdamW
optimizer=AdamW(model.parameters(),lr=2e-5)

In [20]:
from torch.utils.data import DataLoader
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=16)

In [ ]:
EPOCHS = 5
model.train()

from tqdm import tqdm

for epoch in range(EPOCHS):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1}, Avg Loss: {total_loss / len(train_loader):.4f}")


Epoch 1:  25%|████████████▍                                     | 622/2500 [3:00:01<417:32:08, 800.39s/it, loss=0.0864]